**ФИО студента**: Дьякова Елизавета Владиславовна

**Группа**: S4102

**Список выполненных пунктов задания**: все задачи

## **Задача 3. Реализация обхода в ширину из нескольких стартовых вершин (Multiple-Source BFS)**

In [ ]:
pip install python-graphblas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.0/353.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 66.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import graphblas as gb
from graphblas import Matrix, Vector, Scalar
from graphblas import dtypes
from graphblas import unary, binary, monoid, semiring
import time
from scipy.io import mmread
import os
import random

Везде считаем, что вершины графа занумерованы подряд с нуля.

Используя python-graphblas реализовать функцию обхода ориентированного графа (MSBFS-Levels) в ширину из нескольких заданных стартовых вершин.

*   Функция принимает представление графа, удобное для неё (загрузка, конвертация реализованы отдельно) и массив номеров стартовых вершин.
*   Функция возвращает массив пар: стартовая вершина, и массив (levels), где для каждой вершины указано, на каком уровне она достижима из этой стартовой. Стартовая вершина достижима на нулевом уровне, если вершина не достижима, то значение соответствующей ячейки сделайте равной $-1$.


In [ ]:
def calc_msbfs_levels(A: Matrix, sources):
  results = []
  n = A.nrows

  for s in sources:
    result = Vector(dtypes.INT32, n)
    f = Vector(bool, n)
    f[s] << True
    level = 0

    while f.nvals > 0:
      result(mask=f.V) << level
      f(~result.S, replace=True) << f.vxm(A, semiring.lor_land)
      level += 1

    levels = []

    for i in range(n):
      if i in result:
        levels.append(int(result[i]))
      else:
        levels.append(-1)

    results.append((s, levels))

  return results

Используя python-graphblas реализовать функцию обхода ориентированного графа (MSBFS-Parents) в ширину из нескольких заданных стартовых вершин.

*   Функция принимает представление графа, удобное для неё (загрузка, конвертация реализованы отдельно) и массив номеров стартовых вершин.
*   Функция возвращает массив пар: стартовая вершина, и массив (parents), где для каждой вершины графа указано, из какой вершины мы пришли в эту по кратчайшему пути из стартовой вершины. При этом для самой стартовой вершины такое значение взять равное $-1$, а для недостижимых вершин взять равное $-2$. При наличии нескольких возможных значений в массивах parents брать наименьшее.


In [ ]:
def calc_msbfs_parents(A: Matrix, sources):
    results = []
    n = A.nrows

    index_ramp = Vector(dtypes.UINT64, n)
    index_ramp.build(range(n), range(n))

    for s in sources:
        parents = Vector(dtypes.UINT64, n)
        parents[s] << s

        wavefront = Vector(dtypes.UINT64, n)
        wavefront[s] << 1

        while wavefront.nvals > 0:
            wavefront << index_ramp.ewise_mult(wavefront, binary.first)
            wavefront(~parents.S, replace=True) << wavefront.vxm(A, semiring.min_first)
            parents(binary.plus) << wavefront

        result = []
        for i in range(n):
            if i == s:
                result.append(-1)
            elif i in parents:
                result.append(int(parents[i]))
            else:
                result.append(-2)

        results.append((s, result))

    return results

Добавить тесты для проверки корректности полученных реализаций.

1. Проверим работу алгоритмов на графе с одной стартовой вершиной

In [ ]:
def test_one_point():
    edges = [(0, 1), (0, 2), (1, 3)]
    n = 4
    rows = [u for u, v in edges]
    cols = [v for u, v in edges]
    A = Matrix.from_coo(rows, cols, [True] * len(edges), nrows=n, ncols=n)
    sources = [0]

    # Levels
    res = calc_msbfs_levels(A, [0])
    assert res == [(0, [0, 1, 1, 2])]

    # Parents
    res = calc_msbfs_parents(A, [0])
    assert res == [(0, [-1, 0, 0, 1])]

    print("Тест пройден")

In [ ]:
test_one_point()

Тест пройден


2. Проверим работу алгоритмов для графа с несколькими стартовыми вершинами

In [ ]:
def test_multiple_points():
    edges = [(0, 1), (2, 3)]
    n = 6
    rows = [u for u, v in edges]
    cols = [v for u, v in edges]
    A = Matrix.from_coo(rows, cols, [True] * len(edges), nrows=n, ncols=n)

    # Levels
    res = calc_msbfs_levels(A, [0, 2])
    levels_dict = {s: l for s, l in res}

    assert levels_dict[0] == [0, 1, -1, -1, -1, -1]
    assert levels_dict[2] == [-1, -1, 0, 1, -1, -1]

    # Parents
    res = calc_msbfs_parents(A, [0, 2])
    parents_dict = {s: p for s, p in res}

    assert parents_dict[0] == [-1, 0, -2, -2, -2, -2]
    assert parents_dict[2] == [-2, -2, -1, 2, -2, -2]

    print("Тест пройден")

In [ ]:
test_multiple_points()

Тест пройден


3. Проверим работу алгоритмов для графов с недостижимыми вершинами и выбором минимального родителя

In [ ]:
def test_unreachable_and_min():
    edges = [(0, 1), (0, 2), (1, 3), (2, 3)]
    n = 5
    rows = [u for u, v in edges]
    cols = [v for u, v in edges]
    A = Matrix.from_coo(rows, cols, [True] * len(edges), nrows=n, ncols=n)

    # Levels
    res = calc_msbfs_levels(A, [0])
    assert res == [(0, [0, 1, 1, 2, -1])]

    # Parents
    res = calc_msbfs_parents(A, [0])
    assert res == [(0, [-1, 0, 0, 1, -2])]

    print("Тест пройден")

In [ ]:
test_unreachable_and_min()

Тест пройден


(+2 балла) Провести экспериментальное исследование полученных реализаций на некоторых больших графах в формате Matrix Market с сайта SuiteSparse Matrix Collection и на случайных сгенерированных. При этом описать зависимость времени работы всех полученных реализаций от размеров графа, его степени разреженности, количестве стартовых вершин.

In [ ]:
!gdown 1BvM_4yZxBQcQPPTVNTGQ3HTyM3fg-d0x
!unzip data_graph.zip
!rm -rf data_graph.zip

Downloading...
From (original): https://drive.google.com/uc?id=1BvM_4yZxBQcQPPTVNTGQ3HTyM3fg-d0x
From (redirected): https://drive.google.com/uc?id=1BvM_4yZxBQcQPPTVNTGQ3HTyM3fg-d0x&confirm=t&uuid=15b9934a-bd88-4749-af95-63ecd72a353f
To: /content/data_graph.zip
100% 50.6M/50.6M [00:00<00:00, 254MB/s]
Archive:  data_graph.zip
replace bcircuit/bcircuit/bcircuit.mtx? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
def measure_time(func, A, sources):
    start = time.perf_counter()
    func(A, sources)
    end = time.perf_counter()
    return end - start

In [ ]:
results = []

paths = [
    './epb3/epb3/epb3.mtx',
    './pkustk12/pkustk12/pkustk12.mtx',
    './twotone/twotone/twotone.mtx',
    './hcircuit/hcircuit/hcircuit.mtx',
    './g7jac160/g7jac160/g7jac160.mtx',
    './bcsstk32/bcsstk32/bcsstk32.mtx',
    './bcircuit/bcircuit/bcircuit.mtx',
    './onera_dual/onera_dual/onera_dual.mtx',
    './circuit_4/circuit_4/circuit_4.mtx',
    './ct20stif/ct20stif/ct20stif.mtx']

for path in paths:
    name = os.path.basename(path)

    M = mmread(path)

    rows, cols = M.tocsr().nonzero()
    A = Matrix.from_coo(rows, cols, [True]*len(rows), nrows=M.shape[0], ncols=M.shape[1])

    A = A | A.T

    t_levels = measure_time(calc_msbfs_levels, A, [0])
    t_parents = measure_time(calc_msbfs_parents, A, [0])
    density = A.nvals / (A.shape[0] * A.shape[0])

    results.append((name, A.shape[0], A.nvals, density, t_levels, t_parents))

    print(f"{name}: nodes={A.shape[0]}, edges={A.nvals}, density={density:.6f}, levels={t_levels:.4f}, parents={t_parents:.4f}")

epb3.mtx: nodes=84617, edges=589679, density=0.000082, levels=9.1769, parents=9.8382
pkustk12.mtx: nodes=94653, edges=7512317, density=0.000839, levels=10.1973, parents=9.7397
twotone.mtx: nodes=120750, edges=2077315, density=0.000142, levels=6.4167, parents=7.8628
hcircuit.mtx: nodes=105676, edges=513096, density=0.000046, levels=6.0666, parents=7.1056
g7jac160.mtx: nodes=47430, edges=1070196, density=0.000476, levels=2.5387, parents=2.5806
bcsstk32.mtx: nodes=44609, edges=2014701, density=0.001012, levels=5.4222, parents=4.1372
bcircuit.mtx: nodes=68902, edges=375558, density=0.000079, levels=4.5350, parents=5.6575
onera_dual.mtx: nodes=85567, edges=419201, density=0.000057, levels=4.9185, parents=5.9906
circuit_4.mtx: nodes=80209, edges=346446, density=0.000054, levels=3.7398, parents=3.7983
ct20stif.mtx: nodes=52329, edges=2600295, density=0.000950, levels=4.5402, parents=3.7335


In [ ]:
path = './epb3/epb3/epb3.mtx'

M = mmread(path)

rows, cols = M.tocsr().nonzero()
A = Matrix.from_coo(rows, cols, [True]*len(rows), nrows=M.shape[0], ncols=M.shape[1])
A = A | A.T

sources_list = [[0], list(range(3)), list(range(5)), list(range(7)), list(range(10))]
results_sources = []

for sources in sources_list:
    t_levels = measure_time(calc_msbfs_levels, A, sources)
    t_parents = measure_time(calc_msbfs_parents, A, sources)

    results_sources.append((len(sources), t_levels, t_parents))
    print(f"sources={len(sources)}: levels={t_levels:.4f}, parents={t_parents:.4f}")

sources=1: levels=8.1463, parents=10.6393
sources=3: levels=27.2011, parents=27.3648
sources=5: levels=46.4547, parents=45.2254
sources=7: levels=64.9763, parents=65.0344
sources=10: levels=91.5429, parents=91.2970


In [ ]:
def generate_random_graphs(n, sparsity):
  rows, cols = [], []
  for i in range(n):
      for j in range(i + 1, n):
          if random.random() < sparsity:
              rows += [i, j]
              cols += [j, i]
  if not rows:
      return gb.Matrix.from_coo([], [], [], nrows=n, ncols=n)
  return gb.Matrix.from_coo(rows, cols, [1.0] * len(rows), nrows=n, ncols=n)

In [ ]:
sizes     = [300, 600, 1000]
sparsities = [0.001, 0.01, 0.1]
sources_counts = [1, 5, 10]

results_random = []

for n in sizes:
    for sparsity in sparsities:
        A = generate_random_graphs(n, sparsity)
        density = A.nvals / (n * n)

        for k in sources_counts:
            sources = list(range(k))

            t_levels = measure_time(calc_msbfs_levels, A, sources)
            t_parents = measure_time(calc_msbfs_parents, A, sources)

            results_random.append((n, sparsity, density, k, t_levels, t_parents))

            print(f"n={n}, sparsity={sparsity}, k={k}: density={density:.6f}, "
                  f"levels={t_levels:.4f}, parents={t_parents:.4f}")

n=300, sparsity=0.001, k=1: density=0.001067, levels=0.0085, parents=0.0074
n=300, sparsity=0.001, k=5: density=0.001067, levels=0.0336, parents=0.0319
n=300, sparsity=0.001, k=10: density=0.001067, levels=0.0639, parents=0.0802
n=300, sparsity=0.01, k=1: density=0.010222, levels=0.0167, parents=0.0270
n=300, sparsity=0.01, k=5: density=0.010222, levels=0.0818, parents=0.0939
n=300, sparsity=0.01, k=10: density=0.010222, levels=0.1725, parents=0.1684
n=300, sparsity=0.1, k=1: density=0.100067, levels=0.0153, parents=0.0157
n=300, sparsity=0.1, k=5: density=0.100067, levels=0.0787, parents=0.0806
n=300, sparsity=0.1, k=10: density=0.100067, levels=0.1586, parents=0.1671
n=600, sparsity=0.001, k=1: density=0.000883, levels=0.0122, parents=0.0125
n=600, sparsity=0.001, k=5: density=0.000883, levels=0.0597, parents=0.0714
n=600, sparsity=0.001, k=10: density=0.000883, levels=0.1367, parents=0.1292
n=600, sparsity=0.01, k=1: density=0.010089, levels=0.0303, parents=0.0302
n=600, sparsity=0.

**Вывод:** В ходе проведенного экспериментального исследования были протестированы реализации алгоритмов MSBFS-Levels и MSBFS-Parents на 10 больших графах (> 44000 вершин) в формате Matrix Market и на случайно сгенерированных графах разного размера, разреженности и числа стартовых вершин.

Анализируя результаты на графах с SuiteSparse Matrix Collection, можно отметить, что MSBFS-Levels и MSBFS-Parents демонстрируют примерно сопоставимые результаты по времени выполнения, но немного быстрее зачастую оказывался MSBFS-Levels. В общем можно отметить, что зачастую, действительно, чем больше граф, тем выше время работы, однако, эта зависимость не линейна, на время также влияет и структура графа. Так установлено, что степень разреженности графа оказывает влияние на время работы алгоритмов: при увеличении плотности время, как правило, увеличивается, так как увеличивается количество ребер, которые нужно обработать.

Также было проанализировано время работы алгоритмов в зависимости от числа стартовых вершин на примере одного из графов. Полученные результаты явно демонстрируют увеличение времени работы MSBFS-Levels и MSBFS-Parents с увеличением стратовых вершин, так как обход графа выполняется независимо для каждой из них.

Результаты, полученные на случайно сгенерированных графах, сопоставимы с результатами на графах с SuiteSparse Matrix Collection, что подтверждает корректность наблюдаемых зависимостей.

(+3 балла) Добавить реализации описанных алгоритмов с использованием других полуколец (any.pair для levels и any.first для parents). Добавить тесты для проверки корректности. Провести экспериментальное исследование со сравнением этих реализаций с первоначальными на различных графах.

In [ ]:
def calc_msbfs_levels_any(A, sources):
    results = []
    n = A.nrows

    for s in sources:
        result = Vector(dtypes.INT32, n)
        f = Vector(bool, n)
        f[s] << True
        level = 0

        while f.nvals > 0:
            result(mask=f.V) << level
            f(~result.S, replace=True) << f.vxm(A, semiring.any_pair)
            level += 1

        levels = []
        for i in range(n):
            if i in result:
                levels.append(int(result[i]))
            else:
                levels.append(-1)

        results.append((s, levels))

    return results

In [ ]:
def calc_msbfs_parents_any(A, sources):
    results = []
    n = A.nrows

    index_ramp = Vector(dtypes.UINT64, n)
    index_ramp.build(range(n), range(n))

    for s in sources:
        parents = Vector(dtypes.UINT64, n)
        parents[s] << s

        wavefront = Vector(dtypes.UINT64, n)
        wavefront[s] << 1

        while wavefront.nvals > 0:
            wavefront << index_ramp.ewise_mult(wavefront, binary.first)
            wavefront(~parents.S, replace=True) << wavefront.vxm(A, semiring.any_first)
            parents(binary.plus) << wavefront

        result = []
        for i in range(n):
            if i == s:
                result.append(-1)
            elif i in parents:
                result.append(int(parents[i]))
            else:
                result.append(-2)

        results.append((s, result))

    return results

Добавим тесты для проверки корректности полученных реализаций.

1. Проверим работу алгоритмов на графе с одной стартовой вершиной

In [ ]:
def test_one_point_new():
    edges = [(0, 1), (0, 2), (1, 3)]
    n = 4
    rows = [u for u, v in edges]
    cols = [v for u, v in edges]
    A = Matrix.from_coo(rows, cols, [True] * len(edges), nrows=n, ncols=n)
    sources = [0]

    # Levels any
    res = calc_msbfs_levels_any(A, [0])
    assert res == [(0, [0, 1, 1, 2])]

    # Parents any
    res = calc_msbfs_parents_any(A, [0])
    assert res == [(0, [-1, 0, 0, 1])]

    print("Тест пройден")

In [ ]:
test_one_point_new()

Тест пройден


2. Проверим работу алгоритмов для графа с несколькими стартовыми вершинами

In [ ]:
def test_multiple_points_new():
    edges = [(0, 1), (2, 3)]
    n = 6
    rows = [u for u, v in edges]
    cols = [v for u, v in edges]
    A = Matrix.from_coo(rows, cols, [True] * len(edges), nrows=n, ncols=n)

    # Levels any
    res = calc_msbfs_levels_any(A, [0, 2])
    levels_dict = {s: l for s, l in res}

    assert levels_dict[0] == [0, 1, -1, -1, -1, -1]
    assert levels_dict[2] == [-1, -1, 0, 1, -1, -1]

    # Parents any
    res = calc_msbfs_parents_any(A, [0, 2])
    parents_dict = {s: p for s, p in res}

    assert parents_dict[0] == [-1, 0, -2, -2, -2, -2]
    assert parents_dict[2] == [-2, -2, -1, 2, -2, -2]

    print("Тест пройден")

In [ ]:
test_multiple_points_new()

Тест пройден


3. Проверим работу алгоритмов для графов с недостижимыми вершинами и выбором минимального родителя

In [ ]:
def test_unreachable_and_min_new():
    edges = [(0, 1), (0, 2), (1, 3), (2, 3)]
    n = 5
    rows = [u for u, v in edges]
    cols = [v for u, v in edges]
    A = Matrix.from_coo(rows, cols, [True] * len(edges), nrows=n, ncols=n)

    # Levels any
    res = calc_msbfs_levels_any(A, [0])
    assert res == [(0, [0, 1, 1, 2, -1])]

    # Parents any
    res = calc_msbfs_parents_any(A, [0])
    _, parents = res[0]

    assert parents[0] == -1
    assert parents[1] == 0
    assert parents[2] == 0
    assert parents[3] in [1, 2]
    assert parents[4] == -2

    print("Тест пройден")

In [ ]:
test_unreachable_and_min_new()

Тест пройден


Проведем экспериментальное исследование на случайно сгенерирвоанных графах

In [ ]:
results_random_compare = []

for n in [100, 300, 600, 1000]:
    for sparsity in [0.001, 0.01, 0.1]:
        A = generate_random_graphs(n, sparsity)

        t_levels = measure_time(calc_msbfs_levels, A, [0])
        t_levels_any = measure_time(calc_msbfs_levels_any, A, [0])

        t_parents = measure_time(calc_msbfs_parents, A, [0])
        t_parents_any = measure_time(calc_msbfs_parents_any, A, [0])

        results_random_compare.append(
            (n, sparsity, t_levels, t_levels_any, t_parents, t_parents_any)
        )

        print(f"n={n}, p={sparsity}: "
              f"levels={t_levels:.4f}, levels_any={t_levels_any:.4f}, "
              f"parents={t_parents:.4f}, parents_any={t_parents_any:.4f}")

n=100, p=0.001: levels=0.0056, levels_any=0.0025, parents=0.0026, parents_any=0.0024
n=100, p=0.01: levels=0.0025, levels_any=0.0028, parents=0.0026, parents_any=0.0025
n=100, p=0.1: levels=0.0062, levels_any=0.0054, parents=0.0059, parents_any=0.0060
n=300, p=0.001: levels=0.0065, levels_any=0.0067, parents=0.0064, parents_any=0.0063
n=300, p=0.01: levels=0.0161, levels_any=0.0148, parents=0.0150, parents_any=0.0147
n=300, p=0.1: levels=0.0155, levels_any=0.0155, parents=0.0206, parents_any=0.0197
n=600, p=0.001: levels=0.0207, levels_any=0.0187, parents=0.0130, parents_any=0.0124
n=600, p=0.01: levels=0.0289, levels_any=0.0346, parents=0.0325, parents_any=0.0324
n=600, p=0.1: levels=0.0282, levels_any=0.0281, parents=0.0288, parents_any=0.0290
n=1000, p=0.001: levels=0.0234, levels_any=0.0260, parents=0.0240, parents_any=0.0246
n=1000, p=0.01: levels=0.0807, levels_any=0.0818, parents=0.0814, parents_any=0.0758
n=1000, p=0.1: levels=0.0893, levels_any=0.0827, parents=0.0764, parents_

**Вывод:** В ходе проведенного экспериментального исследования было проведено сравнение работы алгоритмов MSBFS-Levels и MSBFS-Parents с использованием других полуколец (any.pair для levels и any.first для parents) на случайно сгенерированных графах.

Результаты демонстрируют, что реализации с использованием полуколец any_pair и any_first демонстрируют сопоставимое время выполнения с исходными алгоритмами. В ряде случаев наблюдается небольшое ускорение, но оно не является стабильным и зависит от структуры графа и его параметров.